# Day 17 — Nulls: fill, drop, detect

*100 Days of Polars · Phase 2: Expressions*

- **Post:** [Handling Nulls in Polars: Fill, Drop, Detect](https://python.plainenglish.io/day-17-handling-nulls-in-polars-fill-drop-detect-1628943ab08a)
- **Dataset:** [Olist Brazilian e-commerce](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) — CSVs in `data/raw/`
- **Written for:** Polars 2.x (run from the repo root)

In [34]:
import polars as pl

print("polars", pl.__version__)

polars 2.0.0


Polars has one missing-value marker for every dtype: **null**. (NaN is a different thing: a float value. More on that in the data-types phase.)

In [35]:
orders = pl.read_csv("data/raw/olist_orders_dataset.csv", try_parse_dates=True)
products = pl.read_csv("data/raw/olist_products_dataset.csv")
reviews = pl.read_csv("data/raw/olist_order_reviews_dataset.csv", try_parse_dates=True)

## 1. Detect

In [36]:
orders.null_count()

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,160,1783,2965,0


In [37]:
# Nulls usually mean something. Here: orders that never reached the customer
(
    orders.filter(pl.col("order_delivered_customer_date").is_null())
    .group_by("order_status")
    .len()
    .sort("len", descending=True)
)

order_status,len
str,u32
"""shipped""",1107
"""canceled""",619
"""unavailable""",609
"""invoiced""",314
"""processing""",301
"""delivered""",8
"""created""",5
"""approved""",2


In [38]:
reviews.select(
    has_title=pl.col("review_comment_title").is_not_null().mean().round(3),
    has_message=pl.col("review_comment_message").is_not_null().mean().round(3),
)

has_title,has_message
f64,f64
0.117,0.413


## 2. `len()` vs `count()`

`pl.len()` counts rows. `count()` counts **non-null** values. Mixing them up is a quiet source of wrong ratios.

In [39]:
orders.select(
    rows=pl.len(),
    delivered_dates=pl.col("order_delivered_customer_date").count(),
    approvals=pl.col("order_approved_at").count(),
)

rows,delivered_dates,approvals
u32,u32,u32
99441,96476,99281


## 3. Fill

In [40]:
# With a value (strings are literals here)
products.select(pl.col("product_category_name").fill_null("unknown")).group_by("product_category_name").len().sort("len").tail(3)

product_category_name,len
str,u32
"""moveis_decoracao""",2657
"""esporte_lazer""",2867
"""cama_mesa_banho""",3029


In [41]:
# With a strategy: fills are computed from the column itself
s = pl.Series("x", [1, None, None, 4, None])
pl.DataFrame({
    "x": s,
    "forward": s.fill_null(strategy="forward"),
    "backward": s.fill_null(strategy="backward"),   # trailing null stays: nothing comes after it
    "min": s.fill_null(strategy="min"),
    "max": s.fill_null(strategy="max"),
    "mean": s.fill_null(strategy="mean"),           # check the dtype
    "zero": s.fill_null(strategy="zero"),
    "one": s.fill_null(strategy="one"),
})

x,forward,backward,min,max,mean,zero,one
i64,i64,i64,i64,i64,i64,i64,i64
1,1,1,1,1,1,1,1
null,1,4,1,4,2,0,1
null,1,4,1,4,2,0,1
4,4,4,4,4,4,4,4
null,4,null,1,4,2,0,1


In [42]:
# The "mean" strategy keeps the column's type: Int64 in, Int64 out (2, not 2.5)
s = pl.Series("x", [1, None, None, 4, None])
pl.DataFrame({
    "int_mean": s.fill_null(strategy="mean"),
    "float_mean": s.cast(pl.Float64).fill_null(strategy="mean"),   # cast first for 2.5
})

int_mean,float_mean
i64,f64
1,1.0
2,2.5
2,2.5
4,4.0
2,2.5


In [43]:
# limit = max nulls IN A ROW filled after each real value
s = pl.Series("x", [1, None, None, None, 4, None, None])
pl.DataFrame({
    "x": s,
    "forward_limit_1": s.fill_null(strategy="forward", limit=1),
    "forward_limit_2": s.fill_null(strategy="forward", limit=2),
    "forward": s.fill_null(strategy="forward"),
})

x,forward_limit_1,forward_limit_2,forward
i64,i64,i64,i64
1,1,1,1
null,1,1,1
null,null,1,1
null,null,null,1
4,4,4,4
null,4,4,4
null,null,4,4


**Sort before you forward or backward fill.** Both copy from the row above or below, so the row order *is* the logic.

In [44]:
prices = pl.DataFrame({"day": [3, 1, 4, 2, 5], "price": [None, 10.0, 12.0, None, None]})

print(prices.with_columns(filled=pl.col("price").fill_null(strategy="forward")))              # unsorted: day 2 gets day 4's price
print(prices.sort("day").with_columns(filled=pl.col("price").fill_null(strategy="forward")))  # sorted: correct

shape: (5, 3)
┌─────┬───────┬────────┐
│ day ┆ price ┆ filled │
│ --- ┆ ---   ┆ ---    │
│ i64 ┆ f64   ┆ f64    │
╞═════╪═══════╪════════╡
│ 3   ┆ null  ┆ null   │
│ 1   ┆ 10.0  ┆ 10.0   │
│ 4   ┆ 12.0  ┆ 12.0   │
│ 2   ┆ null  ┆ 12.0   │
│ 5   ┆ null  ┆ 12.0   │
└─────┴───────┴────────┘
shape: (5, 3)
┌─────┬───────┬────────┐
│ day ┆ price ┆ filled │
│ --- ┆ ---   ┆ ---    │
│ i64 ┆ f64   ┆ f64    │
╞═════╪═══════╪════════╡
│ 1   ┆ 10.0  ┆ 10.0   │
│ 2   ┆ null  ┆ 10.0   │
│ 3   ┆ null  ┆ 10.0   │
│ 4   ┆ 12.0  ┆ 12.0   │
│ 5   ┆ null  ┆ 12.0   │
└─────┴───────┴────────┘


In [45]:
# With another column, or the first non-null of several: coalesce
orders.select(
    "order_status",
    last_known_date=pl.coalesce(
        "order_delivered_customer_date",
        "order_delivered_carrier_date",
        "order_approved_at",
        "order_purchase_timestamp",
    ),
).filter(pl.col("order_status") != "delivered").head(5)

order_status,last_known_date
str,datetime[μs]
"""invoiced""",2017-04-13 13:25:17
"""shipped""",2018-06-05 14:32:00
"""invoiced""",2018-08-07 06:15:14
"""processing""",2017-09-03 14:30:09
"""shipped""",2018-01-11 19:39:23


## 4. Drop

In [46]:
print("all rows:                ", products.height)
print("drop_nulls() any column: ", products.drop_nulls().height)
print("drop_nulls(category):    ", products.drop_nulls("product_category_name").height)

all rows:                 32951
drop_nulls() any column:  32340
drop_nulls(category):     32341


In [47]:
# drop_nulls on a DataFrame is an OR: a null in ANY listed column drops the row
print("drop_nulls(weight, length):   ", products.drop_nulls(["product_weight_g", "product_length_cm"]).height)

# "drop only if EVERY column is null" has no built-in: write it as a filter
print("all columns null dropped:     ", products.filter(~pl.all_horizontal(pl.all().is_null())).height)

# filter(is_not_null) is the same as drop_nulls on that column
print("filter is_not_null(category): ", products.filter(pl.col("product_category_name").is_not_null()).height)

drop_nulls(weight, length):    32949
all columns null dropped:      32951
filter is_not_null(category):  32341


In [48]:
# On a column, drop_nulls removes VALUES, so the column gets shorter: fine alone or inside an aggregation
print(products.select(pl.col("product_category_name").drop_nulls()).height)
print(products.select(pl.col("product_weight_g").drop_nulls().mean()))

32341
shape: (1, 1)
┌──────────────────┐
│ product_weight_g │
│ ---              │
│ f64              │
╞══════════════════╡
│ 2276.472488      │
└──────────────────┘


## 5. Aggregations skip nulls

`mean`, `sum`, `min`, `max` ignore nulls. That is usually right, but be explicit about it in reports.

In [49]:
orders.select(
    avg_delivery_days=(pl.col("order_delivered_customer_date") - pl.col("order_purchase_timestamp")).dt.total_days().mean().round(2),
    based_on_orders=pl.col("order_delivered_customer_date").count(),
    out_of=pl.len(),
)

avg_delivery_days,based_on_orders,out_of
f64,u32,u32
12.09,96476,99441


## 6. Gotcha: `!=` drops nulls too

A comparison with null returns null, and `filter` keeps only `True`. So "not equal to X" silently loses every null row.

Same rule as Day 12 (Trap 4): `filter` keeps only `True`, so a null condition drops the row.

In [50]:
not_beauty = products.filter(pl.col("product_category_name") != "beleza_saude").height
not_beauty_incl_null = products.filter(pl.col("product_category_name").ne_missing("beleza_saude")).height
print(not_beauty, not_beauty_incl_null, not_beauty_incl_null - not_beauty, "rows were hiding")

29897 30507 610 rows were hiding


## Try it yourself

1. Which `products` columns share the same null rows? (Hint: compare `is_null()` across columns.)
2. Fill missing `product_weight_g` with the median weight of all products, using `fill_null` with an expression.
3. Count reviews with a message but no title, and the other way round.